In [1]:
import os, glob, shutil
from pathlib import Path

DRIVE_FINETUNED_ROOT = "/content/drive/MyDrive/scGPT_human/finetuned_pdac"

ckpt_dirs = sorted(
    glob.glob(f"{DRIVE_FINETUNED_ROOT}/dev_*"),
    key=os.path.getmtime
)

assert ckpt_dirs, f"No finetuned runs found in {DRIVE_FINETUNED_ROOT}"

BEST_MODEL_DIR = ckpt_dirs[-1]
BEST_MODEL_PATH = f"{BEST_MODEL_DIR}/best_model.pt"

assert os.path.exists(BEST_MODEL_PATH), f"Missing: {BEST_MODEL_PATH}"

# embed_data usually expects args.json too
pretrained_args = "/content/drive/MyDrive/scGPT_human/args.json"
if not os.path.exists(f"{BEST_MODEL_DIR}/args.json") and os.path.exists(pretrained_args):
    shutil.copy(pretrained_args, f"{BEST_MODEL_DIR}/args.json")

print("Using checkpoint dir:", BEST_MODEL_DIR)
print("Using model:", BEST_MODEL_PATH)
print("Files:", os.listdir(BEST_MODEL_DIR))

Using checkpoint dir: /content/drive/MyDrive/scGPT_human/finetuned_pdac/dev_eyeGPT-Apr20-18-31-31
Using model: /content/drive/MyDrive/scGPT_human/finetuned_pdac/dev_eyeGPT-Apr20-18-31-31/best_model.pt
Files: ['id2type.json', 'dev_train_args.yml', 'vocab.json', 'protocol_finetune.py', 'model_e1.pt', 'model_e3.pt', 'model_e5.pt', 'model_e6.pt', 'model_e7.pt', 'run.log', 'best_model.pt', 'args.json']


In [2]:
import scanpy as sc
import numpy as np
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/PDAC_ADJ_Final_Annotated.h5ad"

adata = sc.read_h5ad(DATA_PATH)

TEST_SAMPLES = ["ADJ4", "ADJ5", "PDAC1"]
TRAIN_SAMPLES = [s for s in adata.obs["sample_id"].unique() if s not in TEST_SAMPLES]

adata_train = adata[adata.obs["sample_id"].isin(TRAIN_SAMPLES)].copy()
adata_test = adata[adata.obs["sample_id"].isin(TEST_SAMPLES)].copy()

# scGPT embedding function needs a gene column
adata_train.var["gene_name"] = adata_train.var_names.astype(str)
adata_test.var["gene_name"] = adata_test.var_names.astype(str)

label_col = "scGPT_target_label"

print("Train:", adata_train.shape)
print("Test:", adata_test.shape)
print("Labels:", adata_train.obs[label_col].value_counts().to_dict())

/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


Train: (52481, 36601)
Test: (18593, 36601)
Labels: {'T-Cells': 17253, 'Fibroblasts': 7689, 'Ductal': 7376, 'Macrophages': 4680, 'Acinar': 4438, 'B-Cells': 4172, 'Neutrophils': 2187, 'Endothelial': 1793, 'Stellate': 1105, 'Plasma': 557, 'Mast': 479, 'NK': 377, 'Endocrine': 251, 'Schwann': 124}


In [3]:
from scgpt.tokenizer.gene_tokenizer import GeneVocab

def fixed_insert_token(self, token, index):
    if token in self.get_stoi():
        return

    stoi = self.get_stoi()
    itos = self.get_itos()

    itos.insert(index, token)

    for k, v in list(stoi.items()):
        if v >= index:
            stoi[k] = v + 1

    stoi[token] = index

GeneVocab.insert_token = fixed_insert_token

print("Patched GeneVocab.insert_token")

/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/lib/python3.12/dist-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


Patched GeneVocab.insert_token


In [6]:
import json, os
from collections import OrderedDict
from torchtext.vocab import vocab as torchtext_vocab
from scgpt.tokenizer.gene_tokenizer import GeneVocab

vocab_path = f"{BEST_MODEL_DIR}/vocab.json"

with open(vocab_path, "r") as f:
    token2idx = json.load(f)

print("vocab.json size:", len(token2idx))

@classmethod
def fixed_from_dict(cls, token2idx, default_token=None):
    ordered = OrderedDict(
        sorted(token2idx.items(), key=lambda x: x[1])
    )

    # torchtext vocab wants frequencies, not indices
    ordered_freq = OrderedDict((tok, 1) for tok in ordered.keys())

    obj = cls([])
    obj.vocab = torchtext_vocab(ordered_freq)

    if default_token is not None:
        obj.set_default_token(default_token)

    return obj

GeneVocab.from_dict = fixed_from_dict

print("Patched GeneVocab.from_dict")

vocab.json size: 60697
Patched GeneVocab.from_dict


In [8]:
from scgpt.tokenizer.gene_tokenizer import GeneVocab

vocab = GeneVocab.from_file(f"{BEST_MODEL_DIR}/vocab.json")
print("Loaded vocab size:", len(vocab))

Loaded vocab size: 60697


In [9]:
from scgpt.tasks import embed_data

EMBED_CACHE_DIR = "/content/drive/MyDrive/scGPT_human/finetuned_pdac_embeddings"
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)

train_emb_path = f"{EMBED_CACHE_DIR}/pdac_train_best_scgpt_embeddings.npy"
test_emb_path = f"{EMBED_CACHE_DIR}/pdac_test_best_scgpt_embeddings.npy"
train_label_path = f"{EMBED_CACHE_DIR}/pdac_train_labels.npy"
test_label_path = f"{EMBED_CACHE_DIR}/pdac_test_labels.npy"

print("Embedding train cells...")
adata_train_emb = embed_data(
    adata_train,
    model_dir=BEST_MODEL_DIR,
    gene_col="gene_name",
    batch_size=64,
    return_new_adata=True,
)

print("Embedding test cells...")
adata_test_emb = embed_data(
    adata_test,
    model_dir=BEST_MODEL_DIR,
    gene_col="gene_name",
    batch_size=64,
    return_new_adata=True,
)

X_train = adata_train_emb.X
X_test = adata_test_emb.X
y_train = adata_train.obs[label_col].astype(str).values
y_test = adata_test.obs[label_col].astype(str).values

np.save(train_emb_path, X_train)
np.save(test_emb_path, X_test)
np.save(train_label_path, y_train)
np.save(test_label_path, y_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Embedding train cells...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 821/821 [09:43<00:00,  1.41it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


Embedding test cells...
scGPT - INFO - match 24285/36601 genes in vocabulary of size 60697.


/usr/local/lib/python3.12/dist-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 291/291 [03:25<00:00,  1.41it/s]
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  return fn(*args_all, **kw)


X_train: (52481, 512)
X_test: (18593, 512)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

results = []

# -------------------------
# Logistic Regression
# -------------------------
print("\n" + "="*80)
print("Training Logistic Regression")
print("="*80)

lr = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    random_state=42,
    n_jobs=-1
)

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
bal_acc_lr = balanced_accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr, average="macro", zero_division=0)
rec_lr = recall_score(y_test, y_pred_lr, average="macro", zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, average="macro", zero_division=0)

results.append({
    "model": "Logistic Regression",
    "accuracy": acc_lr,
    "balanced_accuracy": bal_acc_lr,
    "precision_macro": prec_lr,
    "recall_macro": rec_lr,
    "f1_macro": f1_lr
})

print(f"Accuracy: {acc_lr:.4f}")
print(f"Balanced Acc: {bal_acc_lr:.4f}")
print(f"Precision (macro): {prec_lr:.4f}")
print(f"Recall (macro): {rec_lr:.4f}")
print(f"F1 (macro): {f1_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, zero_division=0))


# -------------------------
# KNN
# -------------------------
print("\n" + "="*80)
print("Training KNN")
print("="*80)

knn = KNeighborsClassifier(
    n_neighbors=10,
    metric="euclidean",
    n_jobs=-1
)

knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
bal_acc_knn = balanced_accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn, average="macro", zero_division=0)
rec_knn = recall_score(y_test, y_pred_knn, average="macro", zero_division=0)
f1_knn = f1_score(y_test, y_pred_knn, average="macro", zero_division=0)

results.append({
    "model": "KNN k=10",
    "accuracy": acc_knn,
    "balanced_accuracy": bal_acc_knn,
    "precision_macro": prec_knn,
    "recall_macro": rec_knn,
    "f1_macro": f1_knn
})

print(f"Accuracy: {acc_knn:.4f}")
print(f"Balanced Acc: {bal_acc_knn:.4f}")
print(f"Precision (macro): {prec_knn:.4f}")
print(f"Recall (macro): {rec_knn:.4f}")
print(f"F1 (macro): {f1_knn:.4f}")
print()
print(classification_report(y_test, y_pred_knn, zero_division=0))


# -------------------------
# Results table
# -------------------------
results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df


Training Logistic Regression
Accuracy: 0.9372
Balanced Acc: 0.8710
Precision (macro): 0.9005
Recall (macro): 0.8710
F1 (macro): 0.8831

              precision    recall  f1-score   support

      Acinar       0.96      0.82      0.89      3877
     B-Cells       0.98      1.00      0.99       239
      Ductal       0.89      0.96      0.92      3742
   Endocrine       0.74      0.73      0.74       156
 Endothelial       0.97      0.98      0.97      1454
 Fibroblasts       0.93      0.98      0.95      2361
 Macrophages       0.88      0.99      0.93      1514
        Mast       1.00      0.91      0.95       510
          NK       0.70      0.50      0.58        28
 Neutrophils       1.00      0.96      0.98       181
      Plasma       0.95      0.80      0.87       151
     Schwann       0.72      0.65      0.68        63
    Stellate       0.91      0.93      0.92       462
     T-Cells       0.98      0.99      0.99      3855

    accuracy                           0.94     185

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
1,KNN k=10,0.945571,0.905757,0.914924,0.905757,0.909121
0,Logistic Regression,0.937181,0.871035,0.900530,0.871035,0.883122


In [16]:
EPOCHS   = 60
LR       = 1e-3
PATIENCE = 10

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

n_classes = len(le.classes_)
in_dim = X_train.shape[1]

print("Input dim:", in_dim)
print("Num classes:", n_classes)

X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train_enc, dtype=torch.long)
X_te = torch.tensor(X_test, dtype=torch.float32)

train_dl = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=512,
    shuffle=True,
    pin_memory=(device == "cuda")
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, n_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP(in_dim, [256, 128], n_classes, dropout=0.3).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

best_acc = 0
best_state = None
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for xb, yb in train_dl:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    train_loss = total_loss / len(train_dl.dataset)

    model.eval()
    with torch.no_grad():
        logits = model(X_te.to(device))
        y_pred_enc = logits.argmax(dim=1).cpu().numpy()
        y_pred_labels = le.inverse_transform(y_pred_enc)

    acc_mlp = accuracy_score(y_test, y_pred_labels)
    bal_acc_mlp = balanced_accuracy_score(y_test, y_pred_labels)

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_loss:.4f} | "
        f"acc={acc_mlp:.4f} | "
        f"bal_acc={bal_acc_mlp:.4f}"
    )

    if acc_mlp > best_acc:
        best_acc = acc_mlp
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
model.to(device)

model.eval()
with torch.no_grad():
    logits = model(X_te.to(device))
    y_pred_enc = logits.argmax(dim=1).cpu().numpy()
    y_pred_mlp = le.inverse_transform(y_pred_enc)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
bal_acc_mlp = balanced_accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp, average="macro", zero_division=0)
rec_mlp = recall_score(y_test, y_pred_mlp, average="macro", zero_division=0)
f1_mlp = f1_score(y_test, y_pred_mlp, average="macro", zero_division=0)

results.append({
    "model": "MLP PyTorch",
    "accuracy": acc_mlp,
    "balanced_accuracy": bal_acc_mlp,
    "precision_macro": prec_mlp,
    "recall_macro": rec_mlp,
    "f1_macro": f1_mlp
})

print("\nFinal MLP results")
print(f"MLP accuracy: {acc_mlp:.4f}")
print(f"MLP balanced accuracy: {bal_acc_mlp:.4f}")
print(f"MLP precision macro: {prec_mlp:.4f}")
print(f"MLP recall macro: {rec_mlp:.4f}")
print(f"MLP F1 macro: {f1_mlp:.4f}")
print()
print(classification_report(y_test, y_pred_mlp, zero_division=0))

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df

Device: cuda
Input dim: 512
Num classes: 14
MLP(
  (net): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=14, bias=True)
  )
)
Parameters: 166,798
Epoch 001 | loss=0.3431 | acc=0.9349 | bal_acc=0.8149
Epoch 002 | loss=0.1154 | acc=0.9423 | bal_acc=0.8769
Epoch 003 | loss=0.0944 | acc=0.9464 | bal_acc=0.8954
Epoch 004 | loss=0.0839 | acc=0.9480 | bal_acc=0.8955
Epoch 005 | loss=0.0798 | acc=0.9477 | bal_acc=0.8996
Epoch 006 | loss=0.0746 | acc=0.9491 | bal_acc=0.8901
Epoch 007 | loss=0.0717 | acc=0.9506 | bal_acc=0.8895
Epoch 008 | loss=0.0672 | acc=0.9529 | bal_acc=0.91

,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
2,MLP PyTorch,0.956489,0.912524,0.916770,0.912524,0.914088
1,KNN k=10,0.945571,0.905757,0.914924,0.905757,0.909121
0,Logistic Regression,0.937181,0.871035,0.900530,0.871035,0.883122
